# Summarize context-dependent property coverage

This notebook summarizes the context-dependent properties identified from the final reviewed business change-related statements.

Input files:  
`15000business_statements.xlsx`  
`15000business_change_statements_final.xlsx`

Output file:  
`15000business_context_dependent_property_summary.xlsx`

Context-dependent properties are defined as properties that appear in at least one final reviewed Class 2 change-related entity-property pair. These properties do not express factual change by themselves, but some of their statements can express change when supported by change-related qualifiers.

The notebook calculates how many context-dependent properties were identified, how many entity-property pairs use these properties in the full business sample, and how many of these pairs were finally identified as Class 2 change-related pairs.

This summary is used to show that context-dependent properties may have the potential to express factual change, but only a subset of their entity-property pairs show explicit change-related signals in Wikidata.

In [2]:
import pandas as pd

full_file = "15000business_statements.xlsx"
final_file = "15000business_change_statements_final.xlsx"
output_file = "15000business_context_dependent_property_summary.xlsx"

full = pd.read_excel(
    full_file,
    usecols=["QID", "Property_ID", "Property_Label"]
)

final = pd.read_excel(
    final_file,
    usecols=["QID", "Property_ID", "Property_Label", "change_class"]
)

# Class 2 pairs in the final reviewed change-related dataset
class2_pairs = (
    final[final["change_class"] == "class2"]
    [["QID", "Property_ID", "Property_Label"]]
    .drop_duplicates()
)


context_properties = (
    class2_pairs[["Property_ID", "Property_Label"]]
    .drop_duplicates()
)

# All full-sample pairs using these context-dependent properties
full_context_pairs = (
    full[full["Property_ID"].isin(context_properties["Property_ID"])]
    [["QID", "Property_ID"]]
    .drop_duplicates()
)

summary = pd.DataFrame([{
    "context_dependent_properties": len(context_properties),
    "pairs_using_context_dependent_properties_in_full_sample": len(full_context_pairs),
    "identified_class2_change_related_pairs": len(class2_pairs),
    "identified_percentage": round(len(class2_pairs) / len(full_context_pairs) * 100, 2)
}])

with pd.ExcelWriter(output_file) as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)
    context_properties.to_excel(writer, sheet_name="context_property_list", index=False)

print(f"context_dependent_properties: {len(context_properties)}")
print(
    "pairs_using_context_dependent_properties_in_full_sample: "
    f"{len(full_context_pairs)}"
)
print(f"identified_class2_change_related_pairs: {len(class2_pairs)}")
print(
    "identified_percentage: "
    f"{round(len(class2_pairs) / len(full_context_pairs) * 100, 2)}%"
)
print(f"Saved to {output_file}")

context_dependent_properties: 209
pairs_using_context_dependent_properties_in_full_sample: 85333
identified_class2_change_related_pairs: 5598
identified_percentage: 6.56%
Saved to 15000business_context_dependent_property_summary.xlsx
